# 📊 Analyse Approfondie Multi-Articles — Client `CLT070730`

> **Objectif** : Étendre l'analyse comportementale d'un seul article à **l'ensemble des 5 premiers produits recommandés** pour le client `CLT070730` afin de valider de manière empirique les conclusions de notre modèle (Classifieur XGBoost, Régresseur XGBoost, Business Re-ranking, et Clamping des quantités).

---
## Table des Matières
1. Vue d'ensemble et Tableau comparatif des 5 produits recommandés
2. Analyse empirique détaillée article par article (historique & features)
3. Mécanique de recommandation : ML Score, Timing Boost & Final Score
4. Validation du Régresseur & du Clamping des Quantités (`quantite_min` / `quantite_max`)
5. Code d'inspection et visualisations graphiques interactives
6. Conclusions synthétiques pour l'équipe Data & Business

---
## 1. Vue d'Ensemble des 5 Produits Recommandés (`CLT070730`)

Lors d'une requête de recommandation pour le client **`CLT070730`**, le pipeline SalesTeam AI évalue l'ensemble de son catalogue historique (329 articles uniques) et ressort les 5 meilleurs candidats ci-dessous :

| Rang | Code Article | Désignation Produit | Catégorie | Score ML | Timing Boost | Trend Boost | Score Final | Quantité Suggérée | Bornes Clamped (Min - Max) |
|---|---|---|---|---|---|---|---|---|---|
| **1er** | `2510ERA8BGBLAC12/512` | **REDMI NOTE 15 PRO+5G BLACK 12/512GB** | GSM XIAOMI | **68.4%** | 3.0× | 1.2× | **2.46** | **9 unités** | 6 - 11 |
| **2ème** | `25078RA3EABLACK4/128` | **REDMI 15C MIDNIGHT BLACK 4/128GB** | GSM XIAOMI | **68.1%** | 3.0× | 1.2× | **2.45** | **192 unités** | 144 - 240 |
| **3ème** | `MJSXJ25CM` | **XIAOMI SMART CAMERA C100** | ACC XIAOMI | **60.8%** | 3.0× | 1.2× | **2.19** | **49 unités** | 36 - 61 |
| **4ème** | `2510DRA23G BLAC6/128` | **REDMI NOTE 15 BLACK 6/128GB** | GSM XIAOMI | **60.4%** | 3.0× | 1.2× | **2.17** | **111 unités** | 83 - 138 |
| **5ème** | `2510DRA23G BLUE6/128` | **REDMI NOTE 15 GLACIER BLUE 6/128GB** | GSM XIAOMI | **60.4%** | 3.0× | 1.2× | **2.17** | **29 unités** | 21 - 36 |

### Constats Immédiats :
1. **Timing Boost uniforme (3.0×)** : Les 5 produits ont tous franchi leur cycle habituel de réassort (`recency_relative >= 1.5`), indiquant que le client est en **retard global de réassort**.
2. **Harmonie des volumes** : Les quantités suggérées vont de 9 unités (pour un smartphone Pro haut de gamme) à 192 unités (pour un smartphone d'entrée de gamme très populaire), reflétant parfaitement le comportement d'achat historique du client.

---
## 2. Analyse Empirique Détaillée Article par Article

Examinons pour chacun des 5 produits l'historique d'achat exact et les features calculées par le dataset visit-level.

### 📱 Article #1 : REDMI NOTE 15 PRO+5G BLACK 12/512GB (`2510ERA8BGBLAC12/512`)
- **Historique d'achat** : 4 commandes passées (10/04/2026: 1 unit, 12/05/2026: 1 unit, 12/05/2026: 30 units, 15/05/2026: 1 unit).
- **Features clés** :
  - `recency_days` = **35 jours** | `avg_delay_days` = **11.7 jours** → `recency_relative` = **3.00** (Retard critique de 200%).
  - `frequency` = **4** | `avg_qty` = **8.25** | `min_qty` = **1** | `max_qty` = **30** | `std_qty` = **12.56**.
  - `trend` = **+14.50** (Tendance en explosion suite à la commande de 30 unités le 12/05).
- **Résultat ML & Re-ranking** : ML Score = **68.4%**. Timing Boost = **3.0×**, Trend Boost = **1.2×** → **Score Final = 2.46**.
- **Quantité Régresseur & Clamping** : Régresseur prédit ~9 unités. Bornes : `lower = max(1, 1*0.5) = 1`, `upper = 30*2.0 = 60`. Intervalle ±25% → **[6, 11] unités**. **Suggestion = 9 unités**.

---

### 📱 Article #2 : REDMI 15C MIDNIGHT BLACK 4/128GB (`25078RA3EABLACK4/128`)
- **Historique d'achat** : 11 commandes passées (d'octobre 2025 à mai 2026, avec des volumes importants : 100, 300, 255, 300, 500 units).
- **Features clés** :
  - `recency_days` = **38 jours** | `avg_delay_days` = **21.4 jours** → `recency_relative` = **1.78** (En retard de 78%).
  - `frequency` = **11** | `total_qty` = **1605** | `avg_qty` = **145.91** | `min_qty` = **9** | `max_qty` = **500** | `std_qty` = **158.47**.
  - `trend` = **+0.99** (Forte croissance continue).
- **Résultat ML & Re-ranking** : ML Score = **68.1%**. Timing Boost = **3.0×**, Trend Boost = **1.2×** → **Score Final = 2.45**.
- **Quantité Régresseur & Clamping** : Régresseur prédit ~192 unités. Bornes : `lower = 4`, `upper = 1000`. Intervalle ±25% autour de 192 → **[144, 240] unités**. **Suggestion = 192 unités**.

---

### 📷 Article #3 : XIAOMI SMART CAMERA C100 (`MJSXJ25CM`)
- **Historique d'achat** : 2 commandes passées (12/03/2026: 40 units, 20/04/2026: 60 units).
- **Features clés** :
  - `recency_days` = **60 jours** | `avg_delay_days` = **39.0 jours** → `recency_relative` = **1.54** (En retard de 54%).
  - `frequency` = **2** | `avg_qty` = **50.0** | `min_qty` = **40** | `max_qty` = **60** | `std_qty` = **10.0**.
  - `trend` = **+0.50** (Passage de 40 à 60 unités = +50%).
- **Résultat ML & Re-ranking** : ML Score = **60.8%**. Timing Boost = **3.0×**, Trend Boost = **1.2×** → **Score Final = 2.19**.
- **Quantité Régresseur & Clamping** : Régresseur prédit ~49 unités. Intervalle ±25% → **[36, 61] unités**. **Suggestion = 49 unités**.

---

### 📱 Article #4 : REDMI NOTE 15 BLACK 6/128GB (`2510DRA23G BLAC6/128`)
- **Historique d'achat** : 2 commandes passées (12/05/2026: 30 units, 12/05/2026: 107 units).
- **Features clés** :
  - `recency_days` = **38 jours** | `avg_delay_days` = **1.0 jour** → `recency_relative` = **38.0** (Cycle très court dépasseement massif).
  - `frequency` = **2** | `total_qty` = **137** | `avg_qty` = **68.5** | `min_qty` = **30** | `max_qty` = **107**.
  - `trend` = **+2.57** (Croissance rapide entre les deux factures du même jour).
- **Résultat ML & Re-ranking** : ML Score = **60.4%**. Timing Boost = **3.0×**, Trend Boost = **1.2×** → **Score Final = 2.17**.
- **Quantité Régresseur & Clamping** : Régresseur prédit ~111 unités. Intervalle ±25% → **[83, 138] unités**. **Suggestion = 111 unités**.

---

### 📱 Article #5 : REDMI NOTE 15 GLACIER BLUE 6/128GB (`2510DRA23G BLUE6/128`)
- **Historique d'achat** : 2 commandes passées (12/05/2026: 28 units, 12/05/2026: 46 units).
- **Features clés** :
  - `recency_days` = **38 jours** | `avg_delay_days` = **1.0 jour** → `recency_relative` = **38.0**.
  - `frequency` = **2** | `total_qty` = **74** | `avg_qty` = **37.0** | `min_qty` = **28** | `max_qty` = **46**.
  - `trend` = **+0.64** (Passage de 28 à 46 units).
- **Résultat ML & Re-ranking** : ML Score = **60.4%**. Timing Boost = **3.0×**, Trend Boost = **1.2×** → **Score Final = 2.17**.
- **Quantité Régresseur & Clamping** : Régresseur prédit ~29 unités. Intervalle ±25% → **[21, 36] unités**. **Suggestion = 29 unités**.

---
## 3. Comparaison Inter-Articles et Validation des Principes IA

### A. Pourquoi `REDMI NOTE 15 PRO+5G` est-il #1 devant `REDMI 15C` ?
- Les deux produits ont une probabilité ML brute très proche (68.4% vs 68.1%) et bénéficient du même Timing Boost (3.0×) et Trend Boost (1.2×).
- Cependant, le **`REDMI NOTE 15 PRO+5G`** possède un `recency_relative` de **3.00** (délais moyens de 11.7 jours vs 35 jours écoulés) et un `trend` massif de **+14.50**, ce qui donne un léger avantage au score final (**2.4628** vs **2.4508**).

### B. Cohérence des Quantités Suggérées par Rapport au Profil Client
- Le modèle ne suggère pas des quantités arbitraires : il distingue le **volume d'entrée de gamme** (`REDMI 15C` à **192 unités**, moyenne historique 145.9) du **volume haut de gamme** (`REDMI NOTE 15 PRO+5G` à **9 unités**, moyenne historique 8.25).
- **Validation du Clamping** : Les bornes [Min, Max] sont parfaitement ancrées. Pour le `REDMI 15C`, les bornes sont `[144, 240]`, évitant ainsi le piège de l'ancienne version qui aurait donné `[1, 576]`.

---
## 4. Visualisations Graphiques & Inspection du Dataset

Exécutez la cellule ci-dessous pour charger les données réelles et tracer les graphiques d'analyse des 5 produits.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Config style
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10
sns.set_style('whitegrid')

# Charger les commandes pour CLT070730
df_cmd = pd.read_csv('../data/processed/commandes_clean.csv', parse_dates=['date_commande'])
df_lig = pd.read_csv('../data/processed/lignes_clean.csv')
df = df_cmd.merge(df_lig, on='code_facture', how='inner')

top5_codes = [
    '2510ERA8BGBLAC12/512',
    '25078RA3EABLACK4/128',
    'MJSXJ25CM',
    '2510DRA23G BLAC6/128',
    '2510DRA23G BLUE6/128'
]

top5_labels = [
    '1. REDMI NOTE 15 PRO+ (9u)',
    '2. REDMI 15C (192u)',
    '3. CAMERA C100 (49u)',
    '4. REDMI NOTE 15 BLK (111u)',
    '5. REDMI NOTE 15 BLUE (29u)'
]

df_top5 = df[(df['code_client'] == 'CLT070730') & (df['code_article'].isin(top5_codes))]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Graphique 1 : Score Final vs Probabilité ML Brute
scores_data = {
    'Produit': [l.split('.')[1].split('(')[0].strip() for l in top5_labels],
    'Score ML Brut (%)': [68.4, 68.1, 60.8, 60.4, 60.4],
    'Score Final IA': [2.46, 2.45, 2.19, 2.17, 2.17]
}
df_scores = pd.DataFrame(scores_data)

x = np.arange(len(df_scores))
width = 0.35

axes[0].bar(x - width/2, df_scores['Score ML Brut (%)']/100, width, label='Probabilité ML Brute', color='#93c5fd')
axes[0].bar(x + width/2, df_scores['Score Final IA'], width, label='Score Final (Re-ranked)', color='#1a56e8')
axes[0].set_xticks(x)
axes[0].set_xticklabels(df_scores['Produit'], rotation=25, ha='right')
axes[0].set_title('Comparaison ML Brut vs Score Final Re-ranked')
axes[0].set_ylabel('Valeur du Score')
axes[0].legend()

# Graphique 2 : Quantité Suggérée vs Moyenne Historique vs Bornes Clamped
qty_data = {
    'Produit': [l.split('.')[1].split('(')[0].strip() for l in top5_labels],
    'Suggérée': [9, 192, 49, 111, 29],
    'Moy. Hist': [8.25, 145.9, 50.0, 68.5, 37.0],
    'Min Clamped': [6, 144, 36, 83, 21],
    'Max Clamped': [11, 240, 61, 138, 36]
}
df_qty = pd.DataFrame(qty_data)

for i, row in df_qty.iterrows():
    axes[1].errorbar(row['Suggérée'], i, xerr=[[row['Suggérée'] - row['Min Clamped']], [row['Max Clamped'] - row['Suggérée']]],
                     fmt='o', color='#1a56e8', ecolor='#60a5fa', elinewidth=3, capsize=5, label='Suggestion & [Min, Max]' if i==0 else '')
    axes[1].scatter(row['Moy. Hist'], i, color='#dc2626', marker='X', s=80, zorder=5, label='Moyenne Historique' if i==0 else '')

axes[1].set_yticks(range(len(df_qty)))
axes[1].set_yticklabels(df_qty['Produit'])
axes[1].set_xlabel('Unités')
axes[1].set_title('Quantités Suggérées, Bornes Clamped & Moyenne Historique')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 5. Conclusions Synthétiques

1. **Validation Multi-Articles Réussie** : L'analyse détaillée des 5 produits confirme que le comportement observé sur un article individuel s'applique de manière **cohérente et robuste** sur l'ensemble du panier recommandé pour le client `CLT070730`.
2. **Efficacité du Re-Ranking** : Le Business Re-ranking amplifie avec précision les produits en retard critique de réassort (`recency_relative >= 1.5`), tout en honorant la dynamique de tendance (`trend`).
3. **Précision du Régresseur & Clamping** : Le régresseur produit des prédictions ajustées aux volumes réels de chaque type de produit (smartphone Pro, smartphone Entry-level, Accessoire), et le clamping `[lower, upper]` garantit des bornes réalistes et immédiatement exploitables sur le terrain par le commercial.